# No-metasurface baseline

Same optics chain with the metasurface replaced by identity (free propagation + trainable
soft detectors + linear head only). This is the 3D analog of the repo's 2D no-MS notebook -
it quantifies what the metasurface actually buys. Same splits (seed 0), same margin (0.40),
same metrics as the MS notebook.

In [ ]:
# Setup: run from the rail3D folder with its venv (see SETUP_LAB.md).
# Device resolution: RAIL3D_DEVICE env var wins; otherwise the 'lab' profile picks the
# strongest CUDA card automatically (cuda:auto), so the 5090's index does not matter.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
import numpy as np, torch, matplotlib.pyplot as plt
from dataclasses import replace
from rail3d import config, data3d, losses3d, train3d, optics3d

PROFILE = 'lab'          # switch to 'laptop' only for small tests
device = config.get_device(PROFILE)
config.ensure_dirs()
print('device:', device)

In [ ]:
cfg_none = train3d.TrainConfig(
    run_name='ms3d_none_v1', surface='none', mode='tot',
    n_epoch=1200, batch_size=config.PROFILES[PROFILE].train_batch,
)
hist_none = train3d.train(cfg_none, device=device)

In [ ]:
model_none, _ = train3d.load_trained(cfg_none, device, 'best')
data = train3d.load_all_data(cfg_none, device)
ev_none = train3d.full_evaluation(model_none, data, cfg_none)
print('test:', ev_none['test'])
print('ROC:', {k: v for k, v in ev_none['roc'].items() if k != 'points'})
print(ev_none['confusion'])

## Comparison table (run after the MS notebook)

In [ ]:
import pandas as pd
rows = []
for name, surface in [('ms3d_slm_v1', 'slm'), ('ms3d_metaunit_v1', 'metaunit'), ('ms3d_none_v1', 'none')]:
    try:
        cfg = replace(cfg_none, run_name=name, surface=surface)
        m, st = train3d.load_trained(cfg, device, 'best')
        ev = train3d.full_evaluation(m, data, cfg)
        rows.append({'run': name, **ev['test'],
                     'auc': ev['roc']['auc'], 'tpr@1%fpr': ev['roc']['tpr_at_1pct_fpr']})
    except FileNotFoundError:
        print(name + ': no checkpoint yet - run its notebook first')
pd.DataFrame(rows).set_index('run').round(4) if rows else None